# Notebook 03 — Modelos e Algoritmos

Este notebook apresenta a fundamentação teórica dos modelos e métodos utilizados no projeto. O objetivo é explicar *o que* cada componente faz e *por que* foi escolhido, sem foco em implementação. Os detalhes de código estão nos módulos em `src/` e nos notebooks de resultados.

---

## 1. Formulação do Problema como Ranking

### 1.1. Por que tratar a corrida como ranking

Em Fórmula 1, o resultado de uma corrida é naturalmente um **ranking**: uma ordenação total (ou parcial) dos pilotos. Tratar o problema como classificação binária (vence ou não vence) desperdiça quase toda a informação contida no resultado.

Ao modelar o **ranking completo**, o projeto aproveita:
- todos os duelos posicionais entre pares de pilotos;
- a distância entre posições consecutivas;
- a frequência com que um piloto supera outro ao longo de várias corridas.

### 1.2. Rankings parciais

Um **ranking parcial** contém apenas uma fração dos competidores. No projeto, trabalha-se com os *top-k = 10* pilotos classificados, acrescidos dos pilotos que não completaram a corrida (DNF), adicionados ao final da lista.

Essa escolha é motivada por três razões:
1. A zona de pontuação (top-10) concentra a informação competitiva mais relevante.
2. Posições além do 10.º lugar são mais susceptíveis a eventos aleatórios (punições, estratégias de fim de corrida, voltas a menos).
3. Modelos de ranking tendem a ser mais estáveis quando focados na região de maior densidade informacional.

### 1.3. Representação formal

Cada corrida é representada como um objeto `RaceRecord`:

```python
@dataclass
class RaceRecord:
    season:       int           # Temporada (ex: 2023)
    race:         str           # Nome da etapa (ex: "Bahrain")
    ranking:      list[str]     # Lista ordenada de siglas de pilotos
    n_classified: int           # Número de pilotos no top-k
    n_dnf:        int           # Número de DNFs adicionados ao final
```

Exemplo de ranking parcial com top-3 e dois DNFs:

```
["VER", "PER", "ALO", "HAM", "SAI"]
                       ↑ DNFs
```


## 2. Distância de Kendall

### 2.1. Definição

A **distância de Kendall** entre dois rankings $\sigma$ e $\pi$ conta o número de pares de elementos que aparecem em **ordens opostas** nos dois rankings:

$$d_K(\sigma, \pi) = \left| \{ (i, j) : \sigma(i) < \sigma(j) \text{ e } \pi(i) > \pi(j) \} \right|$$

Para um ranking de $n$ elementos, a distância máxima é $\binom{n}{2} = n(n-1)/2$.

### 2.2. Por que Kendall e não distância de posição

A distância de posição (diferença absoluta entre posições) é sensível a erros de escala: errar a posição de 1.º para 5.º é tratado como equivalente a errar a posição de 16.º para 20.º. A distância de Kendall é sensível à **ordem relativa entre pares**, o que é mais adequado para avaliar rankings de corridas, onde o que importa é quem terminou à frente de quem.

### 2.3. Kendall tau como métrica de avaliação

O **Kendall tau (τ)** normaliza a distância de Kendall para o intervalo $[-1, 1]$:

$$\tau = 1 - \frac{2 \cdot d_K(\sigma, \pi)}{\binom{n}{2}}$$

- $\tau = 1$: ranking previsto idêntico ao real.
- $\tau = 0$: sem correlação ordinal.
- $\tau = -1$: ranking completamente invertido.

Um valor de $\tau = 0.42$ significa que aproximadamente $\frac{1 + 0.42}{2} = 71\%$ dos pares de pilotos foram ordenados corretamente pelo modelo.


## 3. Modelo de Mallows

### 3.1. Ideia central

O **Modelo de Mallows** é uma distribuição de probabilidade sobre rankings. A ideia é que existe um ranking **consenso** $\rho$ — a "corrida típica" de um grupo — e os rankings observados são perturbações em torno desse consenso. Rankings mais próximos do consenso têm probabilidade maior; rankings muito distantes têm probabilidade menor.

A distribuição de Mallows é definida como:

$$P(\sigma \mid \rho, \alpha) \propto \exp(-\alpha \cdot d_K(\sigma, \rho))$$

onde:
- $\sigma$ é o ranking observado;
- $\rho$ é o ranking consenso do cluster;
- $d_K(\sigma, \rho)$ é a distância de Kendall;
- $\alpha \geq 0$ é o parâmetro de concentração — quanto maior, mais os rankings se concentram em torno de $\rho$.

### 3.2. Clusterização de corridas com Mallows

No projeto, o Modelo de Mallows não é usado para modelar uma única distribuição, mas para **clusterizar corridas**. A ideia é que diferentes tipos de circuito (pistas rápidas, circuitos de rua, pistas com alta degradação) geram estruturas de ranking distintas.

O algoritmo de clusterização opera da seguinte forma:

1. Inicializa aleatoriamente a atribuição de cada corrida a um dos $K$ clusters.
2. Calcula o **ranking consenso** de cada cluster como a média ponderada dos rankings atribuídos.
3. Em cada iteração, reatribui cada corrida ao cluster cujo consenso está mais próximo (em distância de Kendall), com probabilidade proporcional a $\exp(-\alpha \cdot d_K(\sigma, \rho_k))$.
4. Recalcula os consensos e repete até convergência.

O resultado é um conjunto de $K$ consensos e uma atribuição de cada corrida histórica a um cluster. No experimento principal, $K = 2$ clusters com 150 iterações e $\alpha = 0.5$.

### 3.3. Uso do cluster para prever uma corrida nova

Ao prever uma corrida futura, o ranking real ainda não está disponível (seria vazamento de informação). A solução adotada é usar o **histórico do circuito**: o cluster mais frequente das corridas anteriores naquele mesmo circuito é usado como contexto da nova corrida.

Se o circuito nunca apareceu antes, usa-se o cluster mais frequente do histórico geral.


## 4. Modelo de Plackett–Luce

### 4.1. Ideia central

O **Modelo de Plackett–Luce** é um modelo probabilístico para rankings baseado em **parâmetros latentes de habilidade**. Cada piloto $i$ possui um parâmetro $\lambda_i > 0$, chamado *skill score*, que representa sua força relativa.

A probabilidade de observar um ranking $\sigma = (\sigma_1, \sigma_2, \ldots, \sigma_n)$ é construída sequencialmente: na posição $k$, o piloto $\sigma_k$ é "sorteado" entre os que ainda não foram posicionados, com probabilidade proporcional ao seu skill score:

$$P(\sigma \mid \boldsymbol{\lambda}) = \prod_{k=1}^{n} \frac{\lambda_{\sigma_k}}{\sum_{j=k}^{n} \lambda_{\sigma_j}}$$

**Exemplo:** com skill scores $\lambda_{\text{VER}} = 3$, $\lambda_{\text{NOR}} = 2$, $\lambda_{\text{LEC}} = 1$:

$$P(\text{VER} > \text{NOR} > \text{LEC}) = \frac{3}{6} \cdot \frac{2}{3} \cdot 1 = \frac{1}{3}$$

### 4.2. Estimação por algoritmo MM ponderado

Os skill scores são estimados a partir dos rankings históricos usando o **algoritmo MM** (*Minorization-Maximization*), uma variante iterativa que converge para a estimativa de máxima verossimilhança.

O algoritmo MM é adaptado para incorporar **pesos por corrida**: corridas com peso maior influenciam mais a estimação dos parâmetros. Isso conecta o Plackett–Luce ao módulo de pesos temporais e regulatórios.

A cada iteração, o score de cada piloto é atualizado como:

$$\lambda_i^{\text{novo}} = \frac{\sum_{\text{corridas}} w_r \cdot \mathbf{1}[i \in \text{ranking}_r]}{\sum_{\text{corridas}} w_r \sum_{k: \sigma_k^r \leq \text{pos}(i)} \frac{1}{\sum_{j \geq k} \lambda_j}}$$

Após cada iteração, os scores são normalizados para somar 1, garantindo comparabilidade.

### 4.3. Skill scores globais e por cluster

O Plackett–Luce é estimado de duas formas:

- **Score global:** estimado sobre todo o histórico ponderado, independente de cluster. Representa a força geral do piloto ao longo do tempo.
- **Score por cluster:** estimado apenas sobre as corridas atribuídas a cada cluster. Representa a força do piloto em contextos específicos.

O score final usado na previsão é uma combinação ponderada:

$$\lambda_i^{\text{combinado}} = (1 - w_c) \cdot \lambda_i^{\text{global}} + w_c \cdot \lambda_i^{\text{cluster}}$$

onde $w_c = 0.7$ quando o cluster tem pelo menos 5 corridas, e reduzido proporcionalmente caso contrário.

### 4.4. Interpretação do skill score

O skill score **não é** a probabilidade direta de vitória. É um parâmetro de força relativa: um piloto com score maior tem maior probabilidade de aparecer acima de outro em qualquer corrida simulada pelo modelo. A probabilidade por posição é obtida separadamente via simulação Monte Carlo (ver seção 6).


## 5. Pesos Temporais e Regulatórios

### 5.1. Motivação

A Fórmula 1 muda substancialmente entre temporadas: regulamentos técnicos, pilotos, equipes e distribuição de competitividade. Uma corrida de 2019 não deve ter o mesmo peso que uma corrida de 2024 ao estimar a força atual dos pilotos.

O projeto incorpora essa dinâmica por meio de pesos que modulam a influência de cada corrida na estimação dos modelos.

### 5.2. Fórmula de peso

O peso de cada corrida é definido como:

$$w_i = \text{era\_weight}(\text{season}_i) \times \exp(-\lambda \cdot \Delta_i)$$

onde:
- $\text{era\_weight}$ é um fator por temporada, refletindo a relevância regulatória;
- $\lambda = 0.015$ é a taxa de decaimento temporal;
- $\Delta_i$ é a distância (em número de corridas) entre a corrida $i$ e a corrida mais recente.

### 5.3. Pesos por era regulatória

| Temporada | Peso de era | Justificativa |
|---|---|---|
| 2019 | 0.40 | Regulamento anterior; contexto competitivo muito diferente |
| 2020 | 0.40 | Temporada encurtada pela pandemia; anomalias no calendário |
| 2021 | 0.50 | Transição; dominância compartilhada entre Mercedes e Red Bull |
| 2022 | 1.00 | Nova era de regulamento técnico (carros de efeito solo) |
| 2023 | 1.00 | Continuidade do regulamento 2022 |
| 2024 | 1.00 | Continuidade; maior relevância para previsões futuras |
| 2025 | 1.00 | Temporada mais recente |

O decaimento exponencial garante que corridas mais antigas, mesmo dentro de uma mesma era, tenham menos influência que as recentes.


## 6. Simulação Monte Carlo para Vetores de Probabilidade

### 6.1. Motivação

O Modelo de Plackett–Luce produz um ranking ordenado (previsão determinística). Para avaliar a qualidade probabilística do modelo, é necessário transformar os skill scores em uma **distribuição de probabilidades por posição** para cada piloto.

### 6.2. Procedimento

O algoritmo de simulação segue o processo generativo do Plackett–Luce:

1. Para cada simulação $s = 1, \ldots, N$ (com $N = 10.000$):
   - Sorteia o piloto da posição 1 com probabilidade $\propto \lambda_i$.
   - Remove esse piloto do pool.
   - Sorteia o piloto da posição 2 com probabilidade $\propto \lambda_i$ entre os restantes.
   - Repete até preencher as $K = 20$ posições.
2. Conta em quantas simulações cada piloto terminou em cada posição.
3. Divide pela quantidade total de simulações para obter probabilidades.

O resultado é um vetor por piloto:

$$\vec{p}_i = [P(i \text{ termina em } 1.º), P(i \text{ termina em } 2.º), \ldots, P(i \text{ termina em } 20.º)]$$

com $\sum_{k=1}^{20} p_{ik} = 1$ para cada piloto $i$.

### 6.3. Propriedades do vetor

- **Erro de estimativa:** com 10.000 simulações, o erro por célula é $\pm \approx 1\%$.
- **Pilotos DNF:** recebem probabilidade concentrada nas últimas posições.
- **Consistência:** a soma das probabilidades de cada posição $k$ sobre todos os pilotos é igual a 1 (exatamente um piloto termina em cada posição).


## 7. Score Rules e Ranked Probability Score (RPS)

### 7.1. O que são score rules

**Score rules** são funções que avaliam a qualidade de uma previsão probabilística após o resultado real ser observado. Uma *strictly proper scoring rule* incentiva o agente a reportar suas probabilidades verdadeiras: qualquer distorção da distribuição real esperada piora a pontuação esperada.

Três score rules importantes:

| Score rule | Fórmula simplificada | Característica |
|---|---|---|
| **Log Score** | $-\log P(\text{resultado real})$ | Penaliza fortemente probabilidades próximas de zero |
| **Brier Score** | $\sum_k (p_k - o_k)^2$ | Erro quadrático entre probabilidades e desfecho binário |
| **RPS** | $\frac{1}{K}\sum_{k=1}^{K}(F_k - O_k)^2$ | Compara CDFs acumuladas; sensível à ordem |

### 7.2. Por que o RPS é adequado para Fórmula 1

O **Ranked Probability Score** compara a **função de distribuição acumulada (CDF)** prevista com a CDF observada:

$$\text{RPS} = \frac{1}{K} \sum_{k=1}^{K} \left( \hat{F}(k) - F^*(k) \right)^2$$

onde:
- $\hat{F}(k) = \sum_{j=1}^{k} \hat{p}_j$ é a CDF prevista até a posição $k$;
- $F^*(k) = \mathbf{1}[\text{posição real} \leq k]$ é a CDF observada (função degrau);
- $K = 20$ é o número de posições.

O RPS é especialmente adequado para Fórmula 1 porque **penaliza mais erros distantes**. Prever que Verstappen terminará em 1.º e ele terminar em 2.º é um erro pequeno. Prever 1.º e ele terminar em 18.º é um erro muito maior. Métricas como acurácia Top-3 não capturam essa diferença.

**Interpretação dos valores:**
- $\text{RPS} = 0$: previsão perfeita.
- $\text{RPS}_{\text{baseline}}$: RPS de um modelo que distribui probabilidade igual entre todos os pilotos ($\approx 0.13$ para 20 posições e 20 pilotos).
- **Ganho** $= \text{RPS}_{\text{baseline}} - \text{RPS}_{\text{modelo}} > 0$: o modelo é mais informativo que o baseline.

### 7.3. RPS Skill Score

A comparação com o baseline é formalizada pelo **RPS Skill Score**:

$$S_{\text{RPS}} = 1 - \frac{\text{RPS}_{\text{modelo}}}{\text{RPS}_{\text{baseline}}}$$

Um valor de $S_{\text{RPS}} = 0.30$ significa que o modelo reduziu o erro probabilístico em 30% em relação ao baseline uniforme. Esse valor é interpretável e permite comparar diferentes versões do modelo.

### 7.4. Por que RPS complementa Top-3 e Kendall tau

| Métrica | O que avalia | Limitação |
|---|---|---|
| Top-3 accuracy | Presença dos pilotos certos no pódio | Ignora confiança e ordem interna |
| Top-5 accuracy | Presença dos pilotos certos no top-5 | Idem |
| Kendall tau | Concordância ordinal do ranking determinístico | Não avalia probabilidades |
| **RPS** | Qualidade da distribuição probabilística sobre posições | Exige vetor de probabilidades |

Um modelo pode ter Kendall tau razoável mas RPS fraco se estiver mal calibrado (ex: confiante em previsões erradas). As três métricas juntas fornecem uma visão complementar e mais completa da qualidade preditiva.


## 8. Features Contextuais da API OpenF1

Além dos rankings históricos, o projeto incorpora features contextuais
coletadas via API OpenF1 para enriquecer a representação de cada corrida.
Estas features são obtidas do endpoint `race_control`, `starting_grid`
e `session_result` e resumidas por sessão de corrida.

---

### 8.1. Features de Eventos de Corrida (race_control)

Todas derivadas do endpoint `race_control`, que registra mensagens oficiais
da direção de prova durante a sessão.

| Feature | Tipo | Descrição |
|---|---|---|
| `sc_count` | int | Número de períodos de **Safety Car** durante a corrida. Cada deployement conta como 1, independente da duração. |
| `vsc_count` | int | Número de períodos de **Virtual Safety Car**. Impõe limite de velocidade sem neutralizar completamente a corrida. |
| `red_flag_count` | int | Número de **bandeiras vermelhas** — interrupções totais da corrida. Geralmente causadas por acidentes graves ou condições de pista. |
| `yellow_flag_count` | int | Número de mensagens de **bandeira amarela** com `flag = "YELLOW"`. Indica perigo local em um setor, exigindo redução de velocidade. Diferente do SC, afeta apenas parte da pista. |

**Por que eventos de corrida importam para o modelo?**

Safety cars, VSCs e bandeiras vermelhas neutralizam vantagens de ritmo acumuladas.
Um piloto que lidera com 10 segundos de vantagem pode ter esse gap zerado por
um safety car. Corridas com muitos desses eventos são estruturalmente mais
imprevisíveis — o modelo não consegue prever o timing desses eventos, mas
o histórico de frequência por circuito é informativo.

---

### 8.2. Feature de Grid de Largada (starting_grid)

| Feature | Tipo | Descrição |
|---|---|---|
| `grid_<SIGLA>` | int | Posição de largada de cada piloto. Ex: `grid_VER = 1` significa que Verstappen largou da pole. Uma coluna por piloto presente na corrida. |

A posição de largada é uma das features pré-corrida mais preditivas em Fórmula 1.
A correlação entre posição de largada e posição de chegada é alta — pilotos que
largam da frente têm vantagem estrutural por evitar tráfego e escolher trajetórias
ideais. Esta feature é **disponível antes da corrida começar**, portanto não
causa vazamento de informação quando usada em previsão.

---

### 8.3. Feature de Abandono (session_result)

| Feature | Tipo | Descrição |
|---|---|---|
| `dnf_<SIGLA>` | int (0/1) | 1 se o piloto não completou a corrida (DNF, DNS ou DSQ); 0 caso contrário. Ex: `dnf_PER = 1`. Uma coluna por piloto presente na corrida. |

O abandono (`dnf_<SIGLA>`) é uma feature **pós-corrida** — só é conhecida após
o resultado final. Ela é usada exclusivamente para análise histórica e para
calcular taxas de abandono por piloto e circuito. **Nunca é usada como feature
de entrada para prever corridas futuras.**

---

### 8.4. Cobertura Temporal

| Período | Cobertura OpenF1 | Features disponíveis |
|---|---|---|
| 2019–2022 | Não disponível | Nenhuma — usadas como neutras/ausentes |
| 2023–2025 | Completa | `sc_count`, `vsc_count`, `red_flag_count`, `yellow_flag_count`, `grid_*`, `dnf_*` |

---

### 8.5. Separação Pré-corrida vs Pós-corrida

Esta distinção é crítica para evitar **vazamento de informação**:

| Feature | Disponibilidade | Pode ser usada para prever? |
|---|---|---|
| `grid_<SIGLA>` | Antes da largada | ✅ Sim |
| `sc_count`, `vsc_count`, `red_flag_count`, `yellow_flag_count` | Durante/após a corrida | ❌ Não para a corrida atual — ✅ como histórico das corridas passadas |
| `dnf_<SIGLA>` | Após a corrida | ❌ Não para a corrida atual — ✅ como histórico das corridas passadas |


## 9. Limitações dos Modelos Atuais

| Limitação | Impacto | Abordagem planejada                                     |
|---|---|---------------------------------------------------------|
| DNF tratado como posição inferior genérica | Não diferencia causas de abandono; penaliza igualmente falhas mecânicas e erros do piloto | Modelo de probabilidade de DNF                          |
| Skill scores estáticos dentro da temporada | Não captura melhoras ou pioras ao longo do campeonato | Features temporais da OpenF1                            |
| Sem features de qualificação ou grid | A posição de largada é altamente preditiva | Grid de largada via OpenF1 (`starting_grid`)            |
| Sem features de clima | Corridas de chuva têm dinâmica completamente diferente | Dados do endpoint `weather` da OpenF1                   |
| $K = 2$ clusters fixos | Pode não capturar toda a diversidade de tipos de corrida | Validação com diferentes valores de $K$                 |
| Baseline apenas uniforme | Comparação insuficiente para avaliar se o modelo é realmente bom | Baselines adicionais: histórico, grid, último resultado |

---